In [2]:
from pymongo import MongoClient
from pymongo.collection import Collection
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

model = LogisticRegression()

all_trained_models = []

all_score = []

all_training_index = []
all_testing_index = []

collection: Collection = MongoClient().wesad.individual

mongo_dict = {
    "individual_scoring": list()
}

for idx in range(15):
    file_to_read = open("features/training_features"+str(idx)+".pickle", "rb")
    X_train = pickle.load(file_to_read)
    file_to_read.close()

    file_to_read = open("features/training_labels"+str(idx)+".pickle", "rb")
    y_train = pickle.load(file_to_read)
    file_to_read.close()
    
    file_to_read = open("features/testing_features"+str(idx)+".pickle", "rb")
    X_test = pickle.load(file_to_read)
    file_to_read.close()

    file_to_read = open("features/testing_labels"+str(idx)+".pickle", "rb")
    y_test = pickle.load(file_to_read)
    file_to_read.close()

    
    model.fit(X_train, y_train)
    print(idx)
    y_pred = model.predict(X_test)
    acc=accuracy_score(y_test, y_pred)
    prec=precision_score(y_test, y_pred, pos_label=2)
    rec=recall_score(y_test, y_pred, pos_label=2)
    f1=f1_score(y_test, y_pred, pos_label=2)
    print(acc, prec, rec, f1)
    
    mongo_dict["individual_scoring"].append({
        "subject_number": idx,
        "acc": acc,
        "prec": prec,
        "rec": rec,
        "f1": f1
    })
    
collection.insert_one(mongo_dict)
